# End-to-End Workflow Demo (post openai-agents migration)

Exercises the **A1 SDK pipeline** for one reviewer question and exposes every intermediate result so you can see exactly what each stage produces:

1. `ChatAgent.screen` — verdict + redacted question
2. `ChatAgent.relevance_check` — in-scope / out-of-scope
3. `Runner.run(orchestrator_agent, ...)` — single SDK call (replaces the legacy plan_team → fan-out → compare → synthesize → balance pipeline)
4. Inspect `result.new_items`:
   - **Tool calls** = the sub-questions the orchestrator dispatched to each specialist / report_agent / general_specialist
   - **Tool outputs** = `SpecialistOutput`, `ReportDraft`, `ReviewReport`
5. `redact_payload` + `ChatAgent.format` → final markdown answer

We bypass `Orchestrator.run` in this notebook and call `Runner.run` directly so we can introspect the SDK's run-history. The agent graph itself comes from the same factories that production uses.

**Architecture refs:** [brainstorm/sdk-wiring.md](../brainstorm/sdk-wiring.md), [brainstorm/architecture-explanation.md](../brainstorm/architecture-explanation.md).

## 0. Setup

In [1]:
%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

import os, sys, json
import nest_asyncio; nest_asyncio.apply()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) \
    if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Pin CWD to PROJECT_ROOT so every relative path used downstream
# (config/data_profiles/, data_tables/, reports/, logs/, ...) resolves
# against the project root regardless of where the kernel started.
os.chdir(PROJECT_ROOT)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'cwd         : {os.getcwd()}')
print(f'OPENAI_API_KEY set: {bool(os.environ.get("OPENAI_API_KEY"))}')

PROJECT_ROOT: /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/AgenticSys_v2
cwd         : /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/AgenticSys_v2
OPENAI_API_KEY set: True


## 1. Knobs

In [2]:
# ─── pick a question, pillar, model ───
QUESTION = "did this customer have any payment returns?"
PILLAR = "credit_risk"
MODEL = "gpt-4.1"

# ─── data source: 'auto' resolves real → simulated → generator ───
DATA_SOURCE = "auto"

# ─── pick a case (or leave None to use the first available) ───
# Run §3 first to see the full list of available case IDs, then set this
# to the one you want to investigate and re-run from §3 onward.
CASE_ID = "366132845011"

# ─── LLM transport: 'openai' (dev) or 'safechain' (private/prod) ───
# Use 'safechain' only if the safechain package is installed.
BACKEND = "openai"

# ─── concurrency: max simultaneous OpenAI requests ───
# Tuning guide (with `max_retries=8` set on AsyncOpenAI in llm/factory.py,
# the SDK absorbs transient 429s by backing off, so you can push this
# higher than the strict TPM-per-request math suggests):
#   gpt-4.1   (30K TPM tier) : 6–12 is a good range
#   gpt-4o    (~150K TPM)    : 12–24
#   gpt-4o-mini              : 24+ is fine
# Higher cap = more parallelism = faster wallclock. If you start hitting
# many retries, ratchet down. The orchestrator fires 2–4 specialist tools
# per turn so caps below 4 effectively serialize the team.
CONCURRENCY_CAP = 4

## 2. Build session clients (the SDK + firewall layer)

This is the **same wiring `main.py` uses**: `FirewallStack` holds the semaphore + retry policy; `build_session_clients(backend=BACKEND)` chooses the LLM transport — either `FirewalledAsyncOpenAI` over `openai.AsyncOpenAI` (dev) or `SafeChainAsyncOpenAI` (private/prod). Either way it returns an `OpenAIChatCompletionsModel` for the Agents SDK; the architecture downstream is identical. `FirewalledChatShim` preserves `ChatAgent`'s legacy `ainvoke` surface.

In [3]:
from llm.firewall_stack import FirewallStack
from llm.factory import build_session_clients, FirewalledChatShim
from logger.event_logger import EventLogger

logger = EventLogger(session_id='nb-demo', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
firewall = FirewallStack(logger=logger, max_retries=2, concurrency_cap=CONCURRENCY_CAP)
clients = build_session_clients(firewall, model_name=MODEL, backend=BACKEND)
chat_llm = FirewalledChatShim(clients)

print(f'backend                   : {clients.backend!r}')
print(f'model                     : {MODEL!r}')
print(f'concurrency_cap           : {CONCURRENCY_CAP}')
print('clients.firewalled_client :', type(clients.firewalled_client).__name__)
print('clients.model              :', type(clients.model).__name__)
print('chat_llm                    :', type(chat_llm).__name__)

backend                   : 'openai'
model                     : 'gpt-4.1'
concurrency_cap           : 4
clients.firewalled_client : FirewalledAsyncOpenAI
clients.model              : OpenAIChatCompletionsModel
chat_llm                    : FirewalledChatShim


## 3. Initialize data layer (gateway → catalog → tools)

`init_tools(gateway, catalog, logger)` sets the module-level state that the `@function_tool`-decorated data tools (`list_available_tables`, `get_table_schema`, `query_table`) read from at call time. The SDK doesn't know about this — it's our lexical-scope dependency injection (see `brainstorm/sdk-wiring.md` §2.4).

In [4]:
from datalayer.catalog import DataCatalog
from datalayer.gateway import LocalDataGateway
from datalayer.generator import DataGenerator
from tools.data_tools import init_tools
from pathlib import Path

_DATA = Path(PROJECT_ROOT) / 'data_tables'
real_dir, sim_dir = _DATA / 'real', _DATA / 'simulated'

def _has(p): return p.is_dir() and any(c.is_dir() for c in p.iterdir())

if DATA_SOURCE == 'real' or (DATA_SOURCE == 'auto' and _has(real_dir)):
    gw = LocalDataGateway.from_case_folders(str(real_dir))
    src = 'real'
elif DATA_SOURCE == 'simulated' or (DATA_SOURCE == 'auto' and _has(sim_dir)):
    gw = LocalDataGateway.from_case_folders(str(sim_dir))
    src = 'simulated'
else:
    gen = DataGenerator(seed=42, cases=20); gen.load_profiles()
    gw = LocalDataGateway.from_generated(gen.generate_all())
    src = 'generator'

# Always pin the catalog to an absolute path so it never depends on CWD.
catalog = DataCatalog(profile_dir=str(Path(PROJECT_ROOT) / 'config' / 'data_profiles'))
assert catalog.list_tables(), (
    f'DataCatalog loaded 0 profiles from {catalog._profile_dir}. '
    f'Check that the path exists and contains *.yaml files.'
)
init_tools(gw, catalog, logger=logger)

available = gw.list_case_ids()
print(f'data source : {src}')
print(f'catalog     : {len(catalog.list_tables())} canonical tables loaded')
print(f'cases       : {len(available)} available')
for i, cid in enumerate(available[:20]):
    print(f'   [{i:>2}] {cid}')
if len(available) > 20:
    print(f'   ... and {len(available) - 20} more')

data source : real
catalog     : 13 canonical tables loaded
cases       : 1 available
   [ 0] 366132845011


### Pick a case to investigate

Choose a case from the list above by setting `CASE_ID` in §1 (Knobs) and **re-running this cell**. Leave `CASE_ID = None` to default to the first available case.

Anything downstream (ground truth panel, the agent run, the per-specialist data preview) reads from this case via the gateway, so changing `CASE_ID` + rerunning from this cell onward switches the whole demo to a different case.

In [5]:
if CASE_ID is None:
    case_id = available[0]
    print(f'CASE_ID is None — defaulting to first: {case_id}')
elif CASE_ID in available:
    case_id = CASE_ID
    print(f'using CASE_ID = {case_id}')
else:
    raise ValueError(
        f'CASE_ID={CASE_ID!r} is not in the available cases. '
        f'Pick one from the list printed above and re-run.'
    )

gw.set_case(case_id)
print(f'active case : {case_id}')
print(f'tables      : {gw.list_tables()}')

using CASE_ID = 366132845011
active case : 366132845011
tables      : ['bureau_data', 'crossbu_cards_data', 'crossbu_merchants_data', 'demographics_data', 'modelling_data', 'payments', 'score_drivers_data', 'spends_data', 'wcc']


### Sync the catalog with this case's real data (run before team construction)

`DataManagerAgent.sync_catalog(case_id)` reconciles the canonical YAML profiles in `config/data_profiles/` against the columns and values actually present in this case's CSVs:

- **auto-aliased** — real column → existing canonical via declared alias / normalized fuzzy match. Adds new alias entries to the YAML when the real header differs from the canonical name.
- **value-drift** — auto-aliased columns whose dtype or categorical vocabulary disagrees with the canonical profile. Sync rewrites `dtype` and `categories` (with `*_pending_review: true` markers); raw-PII vocabularies (6+ digit runs) are flagged but NOT written, keeping unmasked PAN / account ids out of source-controlled YAML.
- **ambiguous** — low-confidence matches; surfaced for human review, not persisted.
- **new columns / tables** — real columns with no canonical home; auto-added with `description_pending: true`.

Running this BEFORE team construction (the `Runner.run` call in §8) means the orchestrator's catalog context — and the `get_table_schema` output every specialist sees — reflects the real data shape. Column-alias resolution, `declared_values` hints, and table-level alias annotations all line up with what `query_table` will actually return on this case.

In [6]:
from agent_factories.data_manager_agent import DataManagerAgent

data_mgr = DataManagerAgent(
    gateway=gw,
    catalog=catalog,
    llm=None,        # sync_catalog itself doesn't need the LLM
    logger=logger,
)

diff = data_mgr.sync_catalog(case_id)

print(f'auto-aliased   : {len(diff.auto_aliased)}')
print(f'value-drift    : {len(diff.value_drift)}')
print(f'ambiguous      : {len(diff.ambiguous)}')
print(f'new columns    : {len(diff.new)}')
print(f'new tables     : {len(diff.new_tables)}')

if diff.value_drift:
    print()
    print('drift entries that updated the canonical YAML:')
    for d in diff.value_drift:
        markers = []
        if d.dtype_drift:
            markers.append(f'dtype→{d.real_dtype}')
        if d.categories_drift:
            n = len(d.observed_categories) if d.observed_categories else 0
            markers.append(f'categories→{n} observed values')
        print(f'   • {d.real_table}.{d.real_col}  ({", ".join(markers)})')

if diff.ambiguous:
    print()
    print('ambiguous (NOT persisted — review):')
    for d in diff.ambiguous:
        cands = ', '.join(f'{c.canonical_table}.{c.canonical_col}' for c in d.candidates[:2])
        print(f'   • {d.real_table}.{d.real_col} → [{cands}]')

auto-aliased   : 135
value-drift    : 1
ambiguous      : 0
new columns    : 0
new tables     : 6

drift entries that updated the canonical YAML:
   • payments.Card Number  (categories→1 observed values)


In [7]:
# ─── DATA CATALOG TREE for this case ───
# Visualize what the AGENTS see when they call list_available_tables() /
# get_table_schema(). The tree is grounded in this case's actual data:
# each branch is a canonical table; if it's aliased to a real CSV name
# (e.g. spends ↔ spends_data) both are shown, plus row count, table
# description, and per-column dtype + description.
#
# Controls:
#   max_cols_per_table  — cap wide tables (model_scores has 50+ cols).
#                         Pass None to show all columns.
#   show_orphans        — append a footer listing canonical tables that
#                         are NOT in this case (so you can tell "missing"
#                         from "not configured").
#
# Use this to debug "data unavailable" complaints — if a specialist says
# a table isn't accessible, check the tree first to see if the canonical
# name maps to a real CSV in the gateway and whether row count is non-zero.
from tools.data_tools import render_catalog_tree

print(render_catalog_tree(max_cols_per_table=8, max_col_desc=70))

# Also show the raw catalog prompt-context (the literal text agents see)
# in a collapsed-by-default form. Uncomment if you want to inspect it:
#
# print()
# print('═══ CATALOG PROMPT-CONTEXT (raw text injected into agents) ═══')
# print(data_mgr.describe_catalog())

data_catalog  (case 366132845011)
├── bureau  ↔  bureau_data  (18 rows)
│       Credit bureau data — one row per external tradeline per case. Each row carries the case-level aggregated scores (FICO, S
│   ├── month  [date]  — Monthly snapshot date for this bureau record (e.g. "Jan-2024"). One r…
│   ├── SBFE Score  [int]  — SBFE Score — business credit score predicting financial delinquency o…
│   ├── Delinquent External Trades  [int]  — Total count of external credit lines on which the customer defaulted.
│   ├── External Delinquency Amount (USD)  [float]  — Total default amount on external credit lines (USD).
│   ├── Total Tradelines  [int]  — Overall count of external credit lines linked to the customer.
│   ├── Overall External Exposure (USD)  [float]  — Total outstanding balance on all external credit lines (USD).
│   ├── Overall External Revolving Balance (USD)  [float]  — Total revolving balance on all external credit lines (USD).
│   ├── Average External Utlization  [float]  — 

## 4. Build the agent graph + ChatAgent

`Orchestrator(...)` with `clients=` constructs the **same agent graph production uses**: 7 specialists + report_agent + general_specialist, all wrapped via `redacting_tool` and registered as tools on `orchestrator_agent`.

In [9]:
from agent_factories.chat_agent import ChatAgent
from agent_factories.helper_tools import build_helper_tools
from config.pillar_loader import PillarLoader
from orchestrator.orchestrator import Orchestrator

pillar_yaml = PillarLoader().load(PILLAR) or {}
helper_tools = build_helper_tools()
chat_agent = ChatAgent(chat_llm, logger, tools=helper_tools)

orch = Orchestrator(
    llm=None, logger=logger, registry=None,
    pillar=PILLAR, pillar_config=pillar_yaml,
    catalog=catalog, gateway=gw, clients=clients,
)

agent = orch.orchestrator_agent
print(f'orchestrator agent: {agent.name}')
print(f'tools registered  : {len(agent.tools)}')
for t in agent.tools:
    print(f'   • {t.name:<22}  — {t.description[:70]}')

orchestrator agent: orchestrator
tools registered  : 9
   • bureau                  — Domain specialist 'bureau'. Bureau domain skill — tradeline analysis, 
   • capacity_afford         — Domain specialist 'capacity_afford'. Capacity & Affordability domain s
   • crossbu                 — Domain specialist 'crossbu'. Cross-BU domain skill — cross-product exp
   • customer_rel            — Domain specialist 'customer_rel'. Customer Relationship domain skill —
   • modeling                — Domain specialist 'modeling'. Modeling domain skill — internal ML risk
   • spend_payments          — Domain specialist 'spend_payments'. Spend & Payments — payment trends,
   • wcc                     — Domain specialist 'wcc'. WCC domain skill — agent-call notes and custo
   • report_agent            — Look up prior curated reports for this case.
   • general_specialist      — Compare specialist outputs and surface contradictions.


## 5. GROUND TRUTH — what's actually in the case

Before running any agents, dump the raw case data so we have a baseline to compare the agents' answers against. **Read this section first**, then come back to it after Stage E to verify the specialists' claims trace to the data.

In [10]:
# Row counts per table
print('═══ tables in this case ═══')
table_rows = {}
for t in gw.list_tables():
    rows = gw.query(t) or []
    table_rows[t] = rows
    print(f'  {t:<25} {len(rows):>4} rows')
print()

═══ tables in this case ═══
  bureau_data                 18 rows
  crossbu_cards_data           3 rows
  crossbu_merchants_data       0 rows
  demographics_data            1 rows
  modelling_data              18 rows
  payments                   186 rows
  score_drivers_data          18 rows
  spends_data               3865 rows
  wcc                         21 rows



## 6. STAGE A — `ChatAgent.screen(question)`

First boundary. ChatAgent decides whether the question is safe + in scope and produces a redacted version (CASE-IDs and 6+ digit runs masked).

In [11]:
import asyncio
from IPython.display import Markdown, display

verdict = asyncio.get_event_loop().run_until_complete(chat_agent.screen(QUESTION))

print(f'passed              : {verdict.passed}')
print(f'reason              : {verdict.reason!r}')
print(f'original question   : {QUESTION!r}')
print(f'redacted question   : {verdict.redacted_question!r}')

screened_question = verdict.redacted_question if verdict.passed else None

passed              : True
reason              : ''
original question   : 'did this customer have any payment returns?'
redacted question   : 'did this customer have any payment returns?'


## 7. STAGE B — `ChatAgent.relevance_check(question)`  (in-scope test)

Independent in-scope/out-of-scope check. The legacy code uses this to decide whether a question even warrants the orchestrator pipeline. Today's `main.py` doesn't call it explicitly (`screen` already covers most cases) — surfacing it here for visibility.

In [12]:
if screened_question is None:
    print('Skipping — screen rejected the question.')
else:
    in_scope, reason = asyncio.get_event_loop().run_until_complete(
        chat_agent.relevance_check(screened_question)
    )
    print(f'in_scope            : {in_scope}')
    print(f'reason              : {reason!r}')

in_scope            : True
reason              : ''


## 8. STAGE C — `Runner.run(orchestrator_agent, …)`

Single SDK call. The orchestrator agent's instructions tell it to call specialists in parallel, then synthesize a `FinalAnswer`. We call `Runner.run` directly here (rather than `Orchestrator.run`) because we want post-run access to `result.new_items` for the next stages.

**This is the only LLM-heavy cell in the notebook** — expect it to run for a few seconds + use API quota.

In [ ]:
from agents import Runner
from agents.exceptions import AgentsException
from agent_factories.app_context import AppContext
from llm.firewall_stack import redact_payload

if screened_question is None:
    raise SystemExit('Question did not pass screen — nothing to run.')

ctx = AppContext(gateway=gw, case_folder=Path(PROJECT_ROOT) / 'reports' / case_id, logger=logger)

screened_question = "Is there an observable ramp-up in spend or merchant concentration in the early window that precedes increases in external debt or payment stress?"
print(screened_question)
try:
    result = asyncio.get_event_loop().run_until_complete(
        Runner.run(orch.orchestrator_agent, screened_question, context=ctx)
    )
    print(f'Runner.run completed.')
    print(f'final_output type : {type(result.final_output).__name__}')
    print(f'new_items count   : {len(result.new_items)}')
except AgentsException as e:
    print(f'Runner.run raised : {type(e).__name__}: {e}')
    print('  → in production, Orchestrator.run would invoke the trace-extraction fallback here')
    result = None

Is there an observable ramp-up in spend or merchant concentration in the early window that precedes increases in external debt or payment stress?


## 9. STAGE D — Sub-questions (tool calls the orchestrator emitted)

Each `ToolCallItem` in `result.new_items` is a tool the orchestrator's LLM decided to call. The tool name tells us **which specialist** got the call; the JSON args tell us **what sub-question** they were asked.

(In the legacy pipeline, this corresponded to `plan_team`'s output — the `TeamAssignment` list. Now it's emergent from the orchestrator agent's reasoning.)

In [12]:
from agents.items import ToolCallItem, ToolCallOutputItem, MessageOutputItem

if result is None:
    print('No result; skipping.')
else:
    tool_calls = [i for i in result.new_items if isinstance(i, ToolCallItem)]
    print(f'{len(tool_calls)} tool call(s) emitted by orchestrator:\n')
    for i, item in enumerate(tool_calls, 1):
        raw = item.raw_item
        name = getattr(raw, 'name', None) or raw.get('name', '?')
        args = getattr(raw, 'arguments', None) or raw.get('arguments', '{}')
        try:
            parsed = json.loads(args) if isinstance(args, str) else args
        except json.JSONDecodeError:
            parsed = {'raw': args}
        sub_q = parsed.get('sub_question') or parsed.get('input') or parsed
        print(f'[{i}] → {name}')
        print(f'      sub-question: {sub_q}')

4 tool call(s) emitted by orchestrator:

[1] → report_agent
      sub-question: Does any report explicitly analyze ramp-up patterns in spend or merchant concentration prior to increases in external debt or payment stress? What evidence or findings are stated?
[2] → spend_payments
      sub-question: Is there an observable ramp-up period in spend (using txn_monthly.spend_total or spends.Amount aggregated by month), specifically before known increases in external debt or indications of payment stress?
[3] → crossbu
      sub-question: Is there an observable ramp-up in merchant concentration (using crossbu_merchants metrics or card concentration data), particularly in the early period before any observed rise in external debt or payment stress?
[4] → bureau
      sub-question: Identify the time window(s) when external debt or signs of payment stress (like increased balances, new delinquencies, or derog marks in bureau data) increase. Flag timing so it can be compared to earlier spend or m

## 10. STAGE E — Specialist findings (tool outputs)

Each `ToolCallOutputItem.output` is the **already-deserialized Pydantic** the wrapped sub-agent returned (`SpecialistOutput`, `ReportDraft`, or `ReviewReport`), post `redact_payload`. Pairing with the corresponding `ToolCallItem` above, you see Q → A for each specialist.

In [13]:
if result is None:
    print('No result; skipping.')
else:
    tool_outputs = [i for i in result.new_items if isinstance(i, ToolCallOutputItem)]
    print(f'{len(tool_outputs)} tool output(s) returned to the orchestrator:\n')
    for i, item in enumerate(tool_outputs, 1):
        raw = item.raw_item
        # Recover the tool name from the output item's raw representation
        name = (raw.get('name') if isinstance(raw, dict) else None) or '?'
        # Try to align with a tool call by call_id
        call_id = (raw.get('call_id') if isinstance(raw, dict) else None)
        if name == '?' and call_id:
            for tc in [x for x in result.new_items if isinstance(x, ToolCallItem)]:
                tcr = tc.raw_item
                tcid = getattr(tcr, 'call_id', None) or (tcr.get('call_id') if isinstance(tcr, dict) else None)
                if tcid == call_id:
                    name = getattr(tcr, 'name', None) or tcr.get('name', '?')
                    break
        out = item.output
        print(f'━━━ [{i}] from {name} ' + '━' * (60 - len(str(name))))
        if hasattr(out, 'model_dump'):
            payload = out.model_dump()
        else:
            payload = out
        print(json.dumps(payload, indent=2, default=str)[:1500])
        print()

4 tool output(s) returned to the orchestrator:

━━━ [1] from report_agent ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"An error occurred while running the tool. Please try again. Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-K7RzTQeygrrqFBvP3vcdrkoI on tokens per min (TPM): Limit 30000, Used 29154, Requested 7742. Please try again in 13.792s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}"

━━━ [2] from spend_payments ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"An error occurred while running the tool. Please try again. Error: Max turns (10) exceeded"

━━━ [3] from crossbu ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{
  "domain": "crossbu",
  "question": "Is there an observable ramp-up in merchant concentration (using crossbu_merchants metrics or card concentration data), particularly in the early period before any observed ris


## 11. STAGE F — `general_specialist`'s review (if called)

Pulled out separately because contradictions / cross-domain insights are useful to highlight. If the orchestrator chose not to consult `general_specialist`, this section will be empty.

In [23]:
if result is None:
    print('No result; skipping.')
else:
    review = None
    for item in result.new_items:
        if not isinstance(item, ToolCallOutputItem):
            continue
        raw = item.raw_item
        call_id = raw.get('call_id') if isinstance(raw, dict) else None
        # match by call_id back to a tool_call named 'general_specialist'
        for tc in [x for x in result.new_items if isinstance(x, ToolCallItem)]:
            tcr = tc.raw_item
            tname = getattr(tcr, 'name', None) or (tcr.get('name') if isinstance(tcr, dict) else None)
            tcid = getattr(tcr, 'call_id', None) or (tcr.get('call_id') if isinstance(tcr, dict) else None)
            if tname == 'general_specialist' and tcid == call_id:
                review = item.output
                break
        if review is not None:
            break

    if review is None:
        print('Orchestrator did not call general_specialist for this question.')
    else:
        payload = review.model_dump() if hasattr(review, 'model_dump') else review
        print('ReviewReport:')
        print(json.dumps(payload, indent=2, default=str))

Orchestrator did not call general_specialist for this question.


## 12. STAGE G — Final answer (post `redact_payload` + `ChatAgent.format`)

This is what `Orchestrator.run` would have returned: the redacted `FinalAnswer` rendered as reviewer-facing markdown.

In [24]:
if result is None:
    print('No result; skipping.')
else:
    final = redact_payload(result.final_output)
    print('═══ FinalAnswer (Pydantic, redacted) ═══')
    print(json.dumps(final.model_dump(), indent=2, default=str)[:2000])
    print()
    print('═══ Rendered via ChatAgent.format ═══')
    display(Markdown(chat_agent.format(final)))

═══ FinalAnswer (Pydantic, redacted) ═══
{
  "answer": "The volatility in the CDSS (Credit Decision Support System) score is primarily driven by dynamic changes in both external and internal credit risk factors, as confirmed by specialist data analysis and prior curated reports. Model-driven evidence points to a small group of powerful variables repeatedly surfacing as top influences on the score: fluctuations in the customer's credit bureau score (cbr_score), spikes in external credit inquiry volume (cust_experian_trans_union_inq_idx), increases in normalized unpaid or overdue amounts (se_no_norm_wtd_pd_unpaid_amt), overall exposure to credit (exp_pif), and the number of active credit trades (tot_cons_comm_trds_g75).\n\nMonth-over-month, these variables swap positions in terms of influence, meaning that changes in the customer's payment behavior, credit-seeking activity, and exposure levels lead to abrupt score movements\u2014sometimes large swings into negative or sharply positive te

## Answer

The volatility in the CDSS (Credit Decision Support System) score is primarily driven by dynamic changes in both external and internal credit risk factors, as confirmed by specialist data analysis and prior curated reports. Model-driven evidence points to a small group of powerful variables repeatedly surfacing as top influences on the score: fluctuations in the customer's credit bureau score (cbr_score), spikes in external credit inquiry volume (cust_experian_trans_union_inq_idx), increases in normalized unpaid or overdue amounts (se_no_norm_wtd_pd_unpaid_amt), overall exposure to credit (exp_pif), and the number of active credit trades (tot_cons_comm_trds_g75).

Month-over-month, these variables swap positions in terms of influence, meaning that changes in the customer's payment behavior, credit-seeking activity, and exposure levels lead to abrupt score movements—sometimes large swings into negative or sharply positive territory. Both narrative and data excerpts note that persistent high utilization, repeated surges in bureau inquiries, and spikes in internal delinquency indicators are especially likely to trigger acute score shifts. Additionally, the loss of historically stabilizing factors—such as steady spend patterns or supplemental card usage—can intensify volatility as risk accelerates toward default. Thus, the CDSS score's volatility is explained by the customer's evolving credit actions and payment discipline, with rapid score swings typically coinciding with new or worsening risk signals in these core features.

## Provenance
- Report coverage: full
- Files consulted: ['driver_exp_0.md', 'modeling_exp_0.md', 'executive_summary_exp_0.md']
- Specialists consulted: ['modeling']

## 13. STAGE H — EventLogger trail (optional)

Hand-emitted JSONL events. Useful for spotting `firewall_rejection`, `tool_call`, `tool_result`, `orchestrator_run_blocked`, etc.

In [16]:
log_path = Path(PROJECT_ROOT) / 'logs' / f'nb-demo.jsonl'
if not log_path.exists():
    print(f'No log at {log_path}')
else:
    from collections import Counter
    events = [json.loads(line) for line in log_path.open() if line.strip()]
    counts = Counter(e.get('event', '?') for e in events)
    print(f'{len(events)} events total in {log_path.name}:')
    for k, v in counts.most_common():
        print(f'  {v:3d}  {k}')
    print()
    print('Last 5 events:')
    for e in events[-5:]:
        print(f"  {e.get('event','?'):<28}  {json.dumps(e.get('payload', {}))[:80]}")

282 events total in nb-demo.jsonl:
   83  tool_call
   83  tool_result
   38  chat_screen_start
   38  chat_screen_done
   21  data_manager_sync_start
   19  data_manager_sync_done

Last 5 events:
  tool_result                   {}
  tool_call                     {}
  tool_result                   {}
  tool_call                     {}
  tool_result                   {}


## 14. Follow-up question (multi-turn)

Multi-turn demo. Set `FOLLOWUP_QUESTION` and run this cell — the orchestrator sees the entire prior conversation (everything from §8) plus your new message, so follow-ups like *"what about the commercial ones?"*, *"show me the dates"*, *"is that unusual?"* route correctly without re-explaining context.

Each re-run of this cell **extends** the conversation: `result` is updated to point at the latest turn, so §9–§12 above will reflect this follow-up turn if you scroll back. To reset to a fresh single-turn conversation, re-run §8.

The mechanics: `result.to_input_list()` is the SDK's canonical way to roll the prior run's full message history forward; we append a new `{role: user, content: ...}` entry and pass the combined list as `Runner.run`'s `input`.

In [41]:
# ─── set a follow-up that builds on the prior turn(s) ───
FOLLOWUP_QUESTION = "show me their total balance for these commercial cards"

if result is None:
    raise SystemExit('No prior `result` — run §8 first to establish conversation state.')

# Screen the follow-up the same way the first question went through ChatAgent.
fu_verdict = asyncio.get_event_loop().run_until_complete(
    chat_agent.screen(FOLLOWUP_QUESTION)
)
print(f'screened           : passed={fu_verdict.passed}')
print(f'redacted follow-up : {fu_verdict.redacted_question!r}')
if not fu_verdict.passed:
    raise SystemExit(f'Screen rejected follow-up: {fu_verdict.reason!r}')

# Build the multi-turn input: prior conversation history (everything `result`
# already has) + the new user message. `to_input_list()` is the SDK's
# canonical way to roll the previous run forward.
followup_input = result.to_input_list() + [
    {'role': 'user', 'content': fu_verdict.redacted_question}
]

try:
    followup_result = asyncio.get_event_loop().run_until_complete(
        Runner.run(orch.orchestrator_agent, followup_input, context=ctx)
    )
    print(f'Runner.run completed.')
    print(f'new_items this turn: {len(followup_result.new_items)}')
except AgentsException as e:
    print(f'Runner.run raised  : {type(e).__name__}: {e}')
    followup_result = None

if followup_result is not None:
    # Chain forward: re-running this cell with a NEW FOLLOWUP_QUESTION will
    # extend the conversation further. Re-run §8 to reset to single-turn.
    # §9–§12 above will now reflect this follow-up turn if you scroll back.
    result = followup_result

    fu_calls = [i for i in result.new_items if isinstance(i, ToolCallItem)]
    print()
    print(f'{len(fu_calls)} tool call(s) on this follow-up turn:')
    for i, item in enumerate(fu_calls, 1):
        raw = item.raw_item
        name = getattr(raw, 'name', None) or raw.get('name', '?')
        args = getattr(raw, 'arguments', None) or raw.get('arguments', '{}')
        try:
            parsed = json.loads(args) if isinstance(args, str) else args
        except json.JSONDecodeError:
            parsed = {'raw': args}
        sub_q = parsed.get('sub_question') or parsed.get('input') or parsed
        print(f'   [{i}] → {name}: {sub_q}')

    final = redact_payload(result.final_output)
    print()
    print('═══ Follow-up FinalAnswer (redacted) ═══')
    display(Markdown(chat_agent.format(final)))

screened           : passed=True
redacted follow-up : 'show me their total balance for these commercial cards'
Runner.run completed.
new_items this turn: 5

2 tool call(s) on this follow-up turn:
   [1] → report_agent: What is the customer's total balance across their commercial cards (portfolio code 'SBS' or other business-indicative portfolio codes)? Provide any aggregate figures, card names, and supporting excerpts.
   [2] → crossbu: What is the total balance across all of the customer's commercial cards, as identified by card_portfolio = 'SBS' or other business-indicative codes in crossbu_cards? Sum the balance column for these cards and return the total value along with which codes were used.

═══ Follow-up FinalAnswer (redacted) ═══


## Answer

The customer's total balance across their commercial cards is $174,897.36. This figure comes from a single card, the 'BUSINESS PLATINUM CARD', which is categorized under the portfolio code 'SBS' (signifying a commercial/business card). No other business-indicative codes were present in the data, and there are no additional commercial cards.

Both the report's narrative and direct specialist data from the crossbu_cards table confirm this total, without contradiction.

## Provenance
- Report coverage: full
- Files consulted: ['crossbu_exp_0.md']
- Specialists consulted: ['crossbu']

In [25]:
if followup_result is None:
    print('No result; skipping.')
else:
    tool_outputs = [i for i in result.new_items if isinstance(i, ToolCallOutputItem)]
    print(f'{len(tool_outputs)} tool output(s) returned to the orchestrator:\n')
    for i, item in enumerate(tool_outputs, 1):
        raw = item.raw_item
        # Recover the tool name from the output item's raw representation
        name = (raw.get('name') if isinstance(raw, dict) else None) or '?'
        # Try to align with a tool call by call_id
        call_id = (raw.get('call_id') if isinstance(raw, dict) else None)
        if name == '?' and call_id:
            for tc in [x for x in followup_result.new_items if isinstance(x, ToolCallItem)]:
                tcr = tc.raw_item
                tcid = getattr(tcr, 'call_id', None) or (tcr.get('call_id') if isinstance(tcr, dict) else None)
                if tcid == call_id:
                    name = getattr(tcr, 'name', None) or tcr.get('name', '?')
                    break
        out = item.output
        print(f'━━━ [{i}] from {name} ' + '━' * (60 - len(str(name))))
        if hasattr(out, 'model_dump'):
            payload = out.model_dump()
        else:
            payload = out
        print(json.dumps(payload, indent=2, default=str)[:1500])
        print()

NameError: name 'followup_result' is not defined